# 02 · Ingeniería de variables

## Predicción del riesgo de rotura de stock en un entorno logístico

En este notebook preparo las variables que utilizaré posteriormente para entrenar los modelos.

Parto del dataset original de ventas y creo variables basadas en:

- la fecha;
- las ventas de días anteriores;
- las medias móviles;
- la tendencia reciente;
- el precio;
- las promociones;
- un plazo de reposición simulado.

El dataset no contiene stock real ni roturas de stock observadas. Por este motivo, el target que voy a crear es una aproximación académica y debe interpretarse como un **riesgo sintético**, no como una rotura real.


## 1. Importación de librerías

Importo las librerías que necesito para trabajar con los datos, realizar cálculos y guardar el resultado.

Mantengo pocas librerías para que el notebook sea fácil de seguir.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

RANDOM_STATE = 42


## 2. Definición de rutas

Utilizo la misma estructura de carpetas que en el notebook anterior.

El dataset procesado se guardará en `data/processed` y los resultados de este notebook se guardarán dentro de `outputs`.


In [ ]:
# Compruebo desde qué carpeta se está ejecutando el notebook.
current_path = Path.cwd()

if current_path.name == "notebooks":
    project_path = current_path.parent
else:
    project_path = current_path

raw_data_path = project_path / "data" / "raw" / "retail_sales.csv"
processed_data_path = project_path / "data" / "processed" / "stockout_dataset.csv"

figures_path = project_path / "outputs" / "figures" / "feature_engineering"
tables_path = project_path / "outputs" / "tables" / "feature_engineering"

processed_data_path.parent.mkdir(parents=True, exist_ok=True)
figures_path.mkdir(parents=True, exist_ok=True)
tables_path.mkdir(parents=True, exist_ok=True)

print("Dataset original:", raw_data_path)
print("Dataset procesado:", processed_data_path)


## 3. Carga y revisión inicial

Cargo el dataset original y compruebo que contiene las columnas necesarias.

Después convierto la fecha y las variables numéricas al tipo adecuado. También ordeno los registros por tienda, producto y fecha, ya que los lags y las medias móviles dependen del orden temporal.


In [ ]:
required_columns = [
    "date",
    "store_id",
    "item_id",
    "sales",
    "price",
    "promo"
]

if not raw_data_path.exists():
    raise FileNotFoundError(
        f"No se encuentra el archivo: {raw_data_path}"
    )

df = pd.read_csv(raw_data_path)

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Faltan columnas obligatorias: {missing_columns}"
    )

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

numeric_columns = [
    "store_id",
    "item_id",
    "sales",
    "price",
    "promo"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

# Elimino únicamente las filas que no tienen los datos básicos necesarios.
df = df.dropna(
    subset=required_columns
).copy()

df = df.sort_values(
    ["store_id", "item_id", "date"]
).reset_index(drop=True)

print("Shape inicial:", df.shape)
display(df.head())


## 4. Comprobación de duplicados

Espero encontrar una sola fila por fecha, tienda y producto.

Si existen duplicados para esa combinación, no continúo porque los lags y las medias móviles podrían calcularse de forma incorrecta.


In [ ]:
key_columns = [
    "date",
    "store_id",
    "item_id"
]

duplicated_rows = df.duplicated(
    subset=key_columns
).sum()

print(
    "Duplicados por fecha, tienda y producto:",
    duplicated_rows
)

if duplicated_rows > 0:
    raise ValueError(
        "Existen duplicados por fecha, tienda y producto. "
        "Debo revisarlos antes de continuar."
    )


## 5. Variables temporales

Creo variables sencillas a partir de la fecha.

Estas variables permiten que el modelo aprenda posibles diferencias entre meses, días de la semana o fines de semana.


In [ ]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["weekday"] = df["date"].dt.dayofweek
df["week"] = df["date"].dt.isocalendar().week.astype(int)

# Creo una variable que indica si el día es sábado o domingo.
df["is_weekend"] = (
    df["weekday"] >= 5
).astype(int)

display(
    df[[
        "date",
        "year",
        "month",
        "week",
        "weekday",
        "is_weekend"
    ]].head()
)


## 6. Variables históricas de ventas

Creo lags para conocer las ventas anteriores de la misma tienda y producto.

Por ejemplo, `sales_lag_7` representa las ventas observadas siete registros antes dentro de la misma serie.

También creo medias móviles usando únicamente ventas anteriores. Para evitar utilizar la venta del mismo día dentro de la media, aplico primero un desplazamiento de un periodo.


In [ ]:
group_columns = [
    "store_id",
    "item_id"
]

sales_group = df.groupby(
    group_columns
)["sales"]

# Creo variables con las ventas de días anteriores.
df["sales_lag_1"] = sales_group.shift(1)
df["sales_lag_7"] = sales_group.shift(7)
df["sales_lag_14"] = sales_group.shift(14)
df["sales_lag_30"] = sales_group.shift(30)


In [ ]:
# Desplazo las ventas un día para que las medias móviles
# solo utilicen información conocida antes de la fecha actual.
shifted_sales = sales_group.shift(1)

df["rolling_mean_7"] = (
    shifted_sales
    .groupby([
        df["store_id"],
        df["item_id"]
    ])
    .transform(
        lambda series: series.rolling(
            window=7,
            min_periods=7
        ).mean()
    )
)

df["rolling_mean_14"] = (
    shifted_sales
    .groupby([
        df["store_id"],
        df["item_id"]
    ])
    .transform(
        lambda series: series.rolling(
            window=14,
            min_periods=14
        ).mean()
    )
)

df["rolling_mean_30"] = (
    shifted_sales
    .groupby([
        df["store_id"],
        df["item_id"]
    ])
    .transform(
        lambda series: series.rolling(
            window=30,
            min_periods=30
        ).mean()
    )
)

df["rolling_std_30"] = (
    shifted_sales
    .groupby([
        df["store_id"],
        df["item_id"]
    ])
    .transform(
        lambda series: series.rolling(
            window=30,
            min_periods=30
        ).std()
    )
)

display(
    df[[
        "store_id",
        "item_id",
        "date",
        "sales",
        "sales_lag_7",
        "rolling_mean_7",
        "rolling_mean_30",
        "rolling_std_30"
    ]].head(35)
)


## 7. Tendencia reciente

Creo una variable sencilla para comparar la demanda reciente con la demanda de un periodo más largo.

Si `trend_7_30` es positiva, la media de los últimos siete días es superior a la media de los últimos treinta días. Esto puede indicar un aumento reciente de la demanda.


In [ ]:
df["trend_7_30"] = (
    df["rolling_mean_7"]
    - df["rolling_mean_30"]
)

display(
    df[[
        "rolling_mean_7",
        "rolling_mean_30",
        "trend_7_30"
    ]].describe().T
)


## 8. Variables de precio y promoción

Mantengo el precio y la promoción actuales porque son datos conocidos en la fecha de predicción.

También creo dos variables sencillas:

- el precio medio de los últimos 30 días;
- el número de días con promoción durante los últimos 7 días.

Estas variables pueden ayudar a explicar cambios de demanda.


In [ ]:
price_group = df.groupby(
    group_columns
)["price"]

shifted_price = price_group.shift(1)

df["rolling_price_mean_30"] = (
    shifted_price
    .groupby([
        df["store_id"],
        df["item_id"]
    ])
    .transform(
        lambda series: series.rolling(
            window=30,
            min_periods=30
        ).mean()
    )
)

promo_group = df.groupby(
    group_columns
)["promo"]

shifted_promo = promo_group.shift(1)

df["promo_days_last_7"] = (
    shifted_promo
    .groupby([
        df["store_id"],
        df["item_id"]
    ])
    .transform(
        lambda series: series.rolling(
            window=7,
            min_periods=7
        ).sum()
    )
)

display(
    df[[
        "price",
        "rolling_price_mean_30",
        "promo",
        "promo_days_last_7"
    ]].head(35)
)


## 9. Simulación del plazo de reposición

El dataset no contiene el plazo real de reposición.

Para poder construir el ejercicio, asigno un `lead_time_days` sencillo según el producto. El resultado siempre estará entre 1 y 7 días y será estable para el mismo producto.

Esta variable es simulada y debe sustituirse por un dato real si el proyecto se utilizara en una empresa.


In [ ]:
# Utilizo el identificador del producto para crear un plazo
# sencillo y reproducible entre 1 y 7 días.
df["lead_time_days"] = (
    df["item_id"].astype(int) % 7
) + 1

display(
    df[[
        "item_id",
        "lead_time_days"
    ]]
    .drop_duplicates()
    .head(15)
)


## 10. Creación de la demanda futura

Para construir el target necesito conocer la demanda de los próximos siete días.

Estas variables futuras se utilizan únicamente para crear la etiqueta. No se utilizarán como variables de entrada de los modelos, porque eso produciría fuga de información.


In [ ]:
future_sales_columns = []

for day_ahead in range(1, 8):
    column_name = f"future_sales_d{day_ahead}"

    df[column_name] = sales_group.shift(
        -day_ahead
    )

    future_sales_columns.append(
        column_name
    )

# Sumo las ventas de los siete días siguientes.
# min_count=7 obliga a tener los siete días disponibles.
df["future_demand_7d"] = df[
    future_sales_columns
].sum(
    axis=1,
    min_count=7
)

display(
    df[[
        "date",
        "sales",
        *future_sales_columns,
        "future_demand_7d"
    ]].head(10)
)


## 11. Creación del target sintético

Como no dispongo de inventario real, creo una estimación sencilla del stock disponible.

Utilizo la media de ventas de los últimos 30 días multiplicada por el plazo de reposición. Después añado un pequeño margen de seguridad basado en la variabilidad de las ventas.

Considero que existe riesgo cuando la demanda futura de siete días supera el stock estimado más el stock de seguridad.

Esta regla es una simulación académica. No representa una rotura de stock observada.


In [ ]:
# Estimo un stock equivalente a la demanda media
# durante el plazo de reposición.
df["stock_estimated"] = (
    df["rolling_mean_30"]
    * df["lead_time_days"]
)

# Creo un margen de seguridad sencillo.
df["safety_stock"] = (
    df["rolling_std_30"]
    * np.sqrt(df["lead_time_days"])
)

df["stockout_risk"] = np.where(
    (
        df["future_demand_7d"].notna()
        & df["stock_estimated"].notna()
        & df["safety_stock"].notna()
    ),
    (
        df["future_demand_7d"]
        > (
            df["stock_estimated"]
            + df["safety_stock"]
        )
    ).astype(int),
    np.nan
)

display(
    df[[
        "future_demand_7d",
        "stock_estimated",
        "safety_stock",
        "stockout_risk"
    ]].head(40)
)


## 12. Selección de variables para el modelado

Selecciono un conjunto reducido de variables que puedo explicar con claridad.

No incluyo las ventas futuras, el stock estimado ni el stock de seguridad porque participan directamente en la creación del target.


In [ ]:
model_features = [
    "sales",
    "price",
    "promo",
    "year",
    "month",
    "week",
    "weekday",
    "is_weekend",
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_14",
    "sales_lag_30",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_30",
    "rolling_std_30",
    "trend_7_30",
    "rolling_price_mean_30",
    "promo_days_last_7",
    "lead_time_days"
]

print(
    f"Número de variables seleccionadas: "
    f"{len(model_features)}"
)

print(model_features)


## 13. Tratamiento de valores nulos

Los primeros registros de cada serie no tienen suficiente historial para calcular los lags y las medias móviles.

Los últimos siete registros tampoco tienen la demanda futura completa.

Elimino únicamente las filas que no tienen alguna variable necesaria para el modelado o para el target.


In [ ]:
required_for_model = (
    model_features
    + ["stockout_risk"]
)

print("Shape antes del filtrado:", df.shape)

valid_rows = df[
    required_for_model
].notna().all(axis=1)

df_model = df.loc[
    valid_rows
].copy()

df_model["stockout_risk"] = (
    df_model["stockout_risk"]
    .astype(int)
)

df_model = df_model.sort_values(
    ["date", "store_id", "item_id"]
).reset_index(drop=True)

print("Shape después del filtrado:", df_model.shape)
print(
    "Filas eliminadas:",
    len(df) - len(df_model)
)


## 14. Distribución del target

Reviso cuántos registros tienen riesgo y qué proporción representan.

Esta comprobación es importante porque un target muy desbalanceado puede afectar a la elección de las métricas durante el entrenamiento.


In [ ]:
target_distribution = (
    df_model["stockout_risk"]
    .value_counts()
    .sort_index()
    .rename_axis("stockout_risk")
    .reset_index(name="count")
)

target_distribution["proportion"] = (
    target_distribution["count"]
    / len(df_model)
)

display(target_distribution)

target_distribution.to_csv(
    tables_path / "target_distribution.csv",
    index=False
)

plt.figure(figsize=(7, 4))

plt.bar(
    target_distribution[
        "stockout_risk"
    ].astype(str),
    target_distribution["count"]
)

plt.title("Distribución del target sintético")
plt.xlabel("Riesgo de rotura")
plt.ylabel("Número de registros")
plt.tight_layout()

plt.savefig(
    figures_path / "target_distribution.png",
    dpi=150
)

plt.show()


## 15. Comprobación de fuga de información

Compruebo que las variables utilizadas para construir el target no aparezcan dentro de `model_features`.

Esta comprobación evita que el modelo aprenda directamente la fórmula utilizada para crear la etiqueta.


In [ ]:
forbidden_features = [
    *future_sales_columns,
    "future_demand_7d",
    "stock_estimated",
    "safety_stock",
    "stockout_risk"
]

leakage_features = [
    feature
    for feature in model_features
    if feature in forbidden_features
]

if leakage_features:
    raise ValueError(
        "Se han encontrado variables con fuga de información: "
        f"{leakage_features}"
    )

print("Control de fuga de información superado.")


## 16. Guardado del dataset procesado

Guardo el dataset completo porque en los siguientes notebooks necesitaré:

- la fecha;
- la tienda;
- el producto;
- las variables del modelo;
- el target;
- algunas variables auxiliares para analizar los resultados.

En el notebook de entrenamiento seleccionaré únicamente las columnas incluidas en `model_features`.


In [ ]:
df_model.to_csv(
    processed_data_path,
    index=False
)

feature_table = pd.DataFrame({
    "feature": model_features
})

feature_table.to_csv(
    tables_path / "model_features.csv",
    index=False
)

print(
    "Dataset procesado guardado en:",
    processed_data_path
)

print("Shape final:", df_model.shape)

display(
    df_model[
        [
            "date",
            "store_id",
            "item_id",
            *model_features,
            "stockout_risk"
        ]
    ].head()
)


## 17. Comprobaciones finales

Realizo unas comprobaciones sencillas antes de cerrar el notebook:

- no debe haber duplicados por fecha, tienda y producto;
- no debe haber nulos en las variables seleccionadas;
- el target solo puede contener 0 y 1;
- el archivo procesado debe haberse creado correctamente.


In [ ]:
assert df_model[
    ["date", "store_id", "item_id"]
].duplicated().sum() == 0

assert df_model[
    required_for_model
].isna().sum().sum() == 0

assert set(
    df_model["stockout_risk"].unique()
).issubset({0, 1})

assert processed_data_path.exists()

print(
    "Todas las comprobaciones finales "
    "se han superado."
)


## 18. Conclusiones

En este notebook he creado un conjunto de variables sencillo y fácil de justificar:

- variables temporales;
- ventas anteriores;
- medias móviles;
- variabilidad de la demanda;
- tendencia reciente;
- precio y promoción;
- plazo de reposición simulado.

También he creado un target sintético para representar el riesgo de rotura de stock.

Las principales limitaciones son:

1. No dispongo de inventario real.
2. No dispongo de lead times reales.
3. No conozco las ventas perdidas.
4. El target depende de una regla simulada.

Por este motivo, los resultados de los modelos deberán interpretarse dentro de un ejercicio académico.

En el siguiente notebook entrenaré varios modelos de clasificación y realizaré una separación temporal entre entrenamiento y test.
